# Research: Short-Term Reversal (1 semaine)

**Serie** : `QuantConnect/research` — carnet de recherche local (yfinance), verdict honnete net de couts.

## Navigation

- [Index de la serie research](README.md)
- soeurs : [research_quality_lowvol](research_quality_lowvol.ipynb) · [research_risk_parity](research_risk_parity.ipynb) · [effet inverse : plus-haut 52 semaines](research_fifty_two_week_high.ipynb)

> Carnet de recherche : le verdict est **falsifiable** (DM + HAC, walk-forward 5-fold chronologique, multi-seed par bruit de rang) et honnete — `BEATS` / `NO BEATS` / `INCONCLUSIVE`, jamais "prometteur".


## Objectifs d'apprentissage

1. Comprendre l'anomalie de **reversal a court terme** : les perdants de la semaine precedente sur-performent les gagnants sur la semaine suivante.
2. Construire un **signal cross-sectionnel** de reversal sur un univers d'ETF liquides (sans Mag7).
3. **Verifier statistiquement** l'edge : walk-forward 5-fold chronologique, multi-seed (bruit de rang), test de Diebold-Mariano a variance HAC (Newey-West) **unilateral**, net de couts de transaction.


## Hypothese

**H** : sur un univers d'ETF liquides, le portefeuille long-short **long du quantile des perdants** de la semaine precedente et **court du quantile des gagnants** genere un edge **net de 10 bps de couts**, mesurable par un test de Diebold-Mariano (p < 0.05, unilaterale, en faveur de la strategy) sur les rets out-of-sample.

Un resultat `NO BEATS` ou `INCONCLUSIVE` est un resultat **valide** : il documente honnetement l'absence d'edge net de couts sur cet univers.


## Prequis

- `yfinance`, `pandas`, `numpy`, `scipy` (env de la serie).
- Notions : backtest long-short, test de Diebold-Mariano, walk-forward chronologique, variance HAC.


In [1]:
"""Imports et configuration de l'univers (ETF liquides, sans Mag7)."""
import numpy as np
import pandas as pd
from scipy import stats

# Univers cross-sectionnel : ETF liquides US, sans Mag7 (critere C).
# (XAWD / XLCO retires : tickers non resolus par yfinance -> 404.)
UNIVERSE = [
    "XLK", "XLF", "XLE", "XLV", "XLI", "XLB", "XLU", "XLP", "XLY", "XLRE",
    "XBI", "XRT", "SMH", "KRE", "ICLN", "TAN", "XME", "XOP", "ARKG", "KOMP",
    "PPA", "IWF",
]

START = "2015-01-01"
END   = "2026-06-30"
COST_BPS = 10          # 10 bps par reequilibrage
LOOKBACK = 5           # fenetre de classement (1 semaine)
HORIZON  = 5           # horizon de detention (1 semaine)
N_QUANT  = 5           # quantiles
N_FOLDS  = 5           # walk-forward 5-fold (chronologique)
SEEDS    = [0, 1, 7, 42, 99]   # >= 4 seeds (critere C)
RANK_NOISE = 0.10      # eto du bruit de rang (z) pour le multi-seed

print(f"Univers declare : {len(UNIVERSE)} ETF | {START} -> {END} | lookback={LOOKBACK}d horizon={HORIZON}d | {len(SEEDS)} seeds")


Univers declare : 22 ETF | 2015-01-01 -> 2026-06-30 | lookback=5d horizon=5d | 5 seeds


In [2]:
"""Telechargement des prix de cloture (yfinance, auto_adjust)."""
import warnings

import yfinance as yf

# yfinance 1.3.0 convertit encore ses cadres avec set_index/rename/drop(inplace=True),
# que pandas 3.1 deprecie (Pandas4Warning, PDEP-8) : ~134 avertissements par
# telechargement, emis par la bibliotheque et non par ce carnet. On filtre cette
# seule categorie, et seulement pendant l'appel ; la correction appartient a
# yfinance, le filtre tombera avec la montee de version (suivi : #28929).
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        category=DeprecationWarning,
        message=r"The inplace keyword in DataFrame\.(set_index|rename|drop|sort_index) is deprecated",
    )
    prices = yf.download(UNIVERSE, start=START, end=END, auto_adjust=True, progress=False)["Close"]

prices = prices.dropna(axis=1, how="any")   # ne garder que les colonnes completes
returns = prices.pct_change().dropna()
N_UNIV = prices.shape[1]
print(f"Prix reels charges : {N_UNIV} ETF (sur {len(UNIVERSE)} declares) x {prices.shape[0]} jours")
print(returns.tail(3))


Prix reels charges : 20 ETF (sur 22 declares) x 2888 jours
Ticker          ARKG      ICLN       IWF       KRE       PPA       SMH  \
Date                                                                     
2026-06-25  0.039658 -0.009277 -0.013119  0.010815  0.003742  0.029018   
2026-06-26  0.032909 -0.038935  0.007959  0.005350 -0.000874 -0.039678   
2026-06-29  0.022448  0.019487  0.024698 -0.005587  0.008628  0.033306   

Ticker           TAN       XBI       XLB       XLE       XLF       XLI  \
Date                                                                     
2026-06-25 -0.004986  0.012558  0.013292  0.009707 -0.005026  0.021697   
2026-06-26 -0.017626  0.025002 -0.004630 -0.004622  0.002245 -0.015859   
2026-06-29  0.013193  0.018857 -0.018217 -0.004829  0.002800  0.008609   

Ticker           XLK       XLP       XLU       XLV       XLY       XME  \
Date                                                                     
2026-06-25  0.008304 -0.005921  0.006807  0.014868 

## 1. Le signal de reversal

Le signal est **cross-sectionnel** : a chaque date de reequilibrage, on classe les ETF par leur retour sur les `LOOKBACK` jours precedents. On est **long** du quantile le plus bas (les perdants recents) et **court** du quantile le plus haut (les gagnants recents). Le portefeuille long-short est reequilibre tous les `HORIZON` jours.

La fonction accepte un `rng` : quand il est fourni, on ajoute un **bruit de rang** (z-score) a la position de chaque ETF dans le classement. C'est ce qui fait varier le portefeuille d'un seed a l'autre — le multi-seed teste si l'edge survit a un re-classement legerement perturbe.


In [3]:
"""Construire les poids long-short de reversal, detenus entre deux reequilibrages.

A chaque date de reequilibrage (tous les HORIZON jours), on classe les ETF
par leur retour sur LOOKBACK jours : poids +1/n dans le quantile bas, -1/n
dans le quantile haut, 0 ailleurs.

Chaque position est ensuite DETENUE jusqu'a la date de reequilibrage suivante
(report borne a HORIZON - 1 lignes). Sans ce report, les poids ne vivent qu'une
seule ligne : le backtest ne detient le portefeuille qu'un jour sur HORIZON,
et paie pourtant l'entree puis la sortie des le lendemain.

Le dernier reequilibrage n'a pas de successeur : sa position arrive a echeance
avant la fin de l'echantillon, qui laisse donc le portefeuille liquide sur ses
dernieres seances. C'est une liquidation de fin de serie, comptee comme un
changement de position de plus.

rng : si fourni, ajoute un bruit de rang de sigma RANK_NOISE (z) — c'est le
levier du multi-seed (le classement de base, rng=None, est deterministe).
"""
def reversal_weights(returns: pd.DataFrame, lookback: int, horizon: int,
                     n_quant: int, rng: "np.random.Generator | None" = None) -> pd.DataFrame:
    weights = pd.DataFrame(0.0, index=returns.index, columns=returns.columns)
    trail = returns.rolling(lookback).sum()
    for i in range(lookback, len(returns) - horizon + 1, horizon):
        row = trail.iloc[i]
        rank = row.rank(pct=True)                      # position dans le classement, en [0, 1]
        if rng is not None:
            rank = rank + rng.normal(0.0, RANK_NOISE, size=len(rank))
        q_lo = 1.0 / n_quant
        q_hi = 1.0 - 1.0 / n_quant
        lo = rank <= q_lo
        hi = rank >= q_hi
        n_lo, n_hi = int(lo.sum()), int(hi.sum())
        if n_lo == 0 or n_hi == 0:
            continue
        weights.iloc[i, lo] = 1.0 / n_lo
        weights.iloc[i, hi] = -1.0 / n_hi
    # Report : 0 signifie « non selectionne a cette date ». On le rend manquant
    # le temps du report, plafonne a HORIZON - 1 lignes, pour que la position
    # couvre exactement les HORIZON jours qui suivent son reequilibrage.
    weights = weights.mask(weights == 0.0).ffill(limit=horizon - 1).fillna(0.0)
    return weights

weights = reversal_weights(returns, LOOKBACK, HORIZON, N_QUANT)   # base deterministe
n_held = int((weights.abs().sum(axis=1) > 0).sum())               # jours ou le portefeuille est detenu
print(f"Poids de base construits : portefeuille detenu {n_held} jours sur {len(weights)}")
weights.abs().sum(axis=1).tail(6)


Poids de base construits : portefeuille detenu 2880 jours sur 2887


Date
2026-06-22    2.0
2026-06-23    2.0
2026-06-24    2.0
2026-06-25    2.0
2026-06-26    0.0
2026-06-29    0.0
dtype: float64

## Exercice 1 -- Calculer le signal pour un horizon different

Reutiliser `reversal_weights` pour un horizon de **10 jours** (plutot que 5) et un classement sur **10 jours** (plutot que 5), puis comparer le nombre de dates de reequilibrage.


In [4]:
# TODO etudiant: calculer les poids pour lookback=10, horizon=10, n_quant=5
# puis afficher le nombre de dates de reequilibrage
weights_ex1 = None  # TODO etudiant
if weights_ex1 is not None:
    print(f"Exercice 1 : {int((weights_ex1.abs().sum(axis=1) > 0).sum())} dates de reequilibrage")
else:
    print("Exercice a completer")


Exercice a completer


## 2. Backtest long-short (net de couts)

Le P&L du portefeuille est le produit des poids et des rets, **net des couts de transaction**. Les couts sont charges a chaque reequilibrage, proportionnellement au turnover (somme des variations absolues de poids).


In [5]:
"""Backtest long-short net de couts.

gross    = (weights.shift(1) * returns).sum(axis=1)   # P&L des positions detenues la veille
turnover = variation de l'envergure des poids
cout     = turnover * COST_BPS / 1e4

Les poids etant reportes d'un reequilibrage au suivant, le turnover est nul
entre deux reequilibrages : un cycle ne paie qu'un aller-retour.
"""
def backtest_reversal(returns: pd.DataFrame, weights: pd.DataFrame, cost_bps: float) -> pd.Series:
    gross = (weights.shift(1) * returns).sum(axis=1)
    turnover = weights.diff().abs().sum(axis=1).fillna(0.0)
    cost = turnover * cost_bps / 1e4
    return (gross - cost).dropna()

strat_ret = backtest_reversal(returns, weights, COST_BPS)

# Controle de la convention de detention : le portefeuille doit produire du P&L
# a chaque jour detenu, et le turnover ne doit bouger qu'aux reequilibrages
# (plus la liquidation de fin de serie). Le decompte brut / couts permet de lire
# ce que le report apporte et ce qu'il coute.
gross_by_day = (weights.shift(1) * returns).sum(axis=1)
turnover_by_day = weights.diff().abs().sum(axis=1).fillna(0.0)
cost_by_day = turnover_by_day * COST_BPS / 1e4
print(f"Jours avec un P&L brut non nul : {int((gross_by_day.abs() > 0).sum())} sur {len(gross_by_day)}")
print(f"Jours avec un turnover non nul : {int((turnover_by_day > 0).sum())} "
      f"(reequilibrages + liquidation finale)")
print(f"Annuelise — P&L brut : {gross_by_day.mean()*252*100:+.2f}% | "
      f"couts : {cost_by_day.mean()*252*100:.2f}%")

# Baseline : equal-weight buy-and-hold de l'univers. Ce n'est PAS la meme
# exposition que la strategy : la baseline est long-only, investie a 100 %,
# la strategy est long-short a exposition nette nulle. Le delta compare donc
# une strategie sans exposition directionnelle a une exposition pleine au
# marche : il mesure l'apport du signal, pas une sur-performance a risque egal.
ew_ret = returns.mean(axis=1)
delta = (strat_ret - ew_ret).dropna()   # positif = la strategy sur-performe
print(f"Ret moyen par periode — strategy : {strat_ret.mean()*100:.4f}% | equal-weight : {ew_ret.mean()*100:.4f}%")
print(f"Annuelise — strategy : {strat_ret.mean()*252*100:.2f}% | equal-weight : {ew_ret.mean()*252*100:.2f}%")
print(f"Delta journalier moyen (strategy - baseline) : {delta.mean()*100:+.4f}%")


Jours avec un P&L brut non nul : 2880 sur 2887
Jours avec un turnover non nul : 577 (reequilibrages + liquidation finale)
Annuelise — P&L brut : +0.66% | couts : 15.01%
Ret moyen par periode — strategy : -0.0569% | equal-weight : 0.0564%
Annuelise — strategy : -14.35% | equal-weight : 14.22%
Delta journalier moyen (strategy - baseline) : -0.1134%


## 3. Walk-forward 5-fold CHRONOLOGIQUE + multi-seed

Pour eviter le sur-ajustement, on valide **out-of-sample** en decoupant la serie en `N_FOLDS` **folds chronologiques contigus** (on ne melange PAS les dates : un melange puis re-trie trierait a nouveau et ne laisserait aucun test vraiment out-of-sample). On compte les folds ou la strategy sur-performe et la moyenne du delta.

Le **multi-seed** injecte un **bruit de rang** de sigma `RANK_NOISE` au classement, d'un seed a l'autre : l'edge doit etre **robuste** a ce re-classement legerement perturbe. La dispersion des deltas par seed donne l'edge en sigmas.


In [6]:
"""Walk-forward 5-fold chronologique + multi-seed par bruit de rang."""
def walk_forward_folds(series: pd.Series, n_folds: int) -> pd.DataFrame:
    """Decoupe la serie en n_folds blocs chronologiques contigus."""
    s = series.dropna()
    n = len(s)
    fold_size = n // (n_folds + 1)
    rows = []
    for k in range(1, n_folds + 1):
        a = fold_size * k
        b = min(a + fold_size, n)
        if b - a < 10:
            continue
        rows.append({"fold": k, "n_oos": b - a, "delta_mean": s.iloc[a:b].mean()})
    return pd.DataFrame(rows)

wf_df = walk_forward_folds(delta, N_FOLDS)
n_positive_folds = int((wf_df["delta_mean"] > 0).sum())
mean_oos_delta = float(delta.mean())
print("Walk-forward 5-fold chronologique :")
print(wf_df.to_string(index=False))
print(f"\nFolds ou la strategy sur-performe : {n_positive_folds}/{len(wf_df)}")
print(f"Moyenne du delta (strategy - baseline) : {mean_oos_delta*100:+.4f}% par periode")


Walk-forward 5-fold chronologique :
 fold  n_oos  delta_mean
    1    481   -0.000919
    2    481   -0.001350
    3    481   -0.001775
    4    481   -0.000297
    5    481   -0.000788

Folds ou la strategy sur-performe : 0/5
Moyenne du delta (strategy - baseline) : -0.1134% par periode


In [7]:
"""Multi-seed : bruit de rang de sigma RANK_NOISE, un seed = un re-classement perturbe."""
seed_deltas = {}
for s in SEEDS:
    rng = np.random.default_rng(s)
    w_s = reversal_weights(returns, LOOKBACK, HORIZON, N_QUANT, rng=rng)
    strat_s = backtest_reversal(returns, w_s, COST_BPS)
    seed_deltas[s] = float((strat_s - ew_ret).dropna().mean())

seed_vals = np.array(list(seed_deltas.values()))
if len(seed_vals) >= 2 and seed_vals.std(ddof=1) > 1e-12:
    edge_sigma = float(seed_vals.mean() / seed_vals.std(ddof=1))
else:
    edge_sigma = float("nan")

print("Delta OOS moyen par seed (strategy - baseline) :")
for s, v in seed_deltas.items():
    print(f"  seed {s:>3} : {v*100:+.4f}%")
print(f"\nEdge cross-seed : {edge_sigma:.2f} sigma  (>= 2 : {edge_sigma >= 2.0})")


Delta OOS moyen par seed (strategy - baseline) :
  seed   0 : -0.1276%
  seed   1 : -0.1109%
  seed   7 : -0.1140%
  seed  42 : -0.1251%
  seed  99 : -0.1234%

Edge cross-seed : -16.43 sigma  (>= 2 : False)


## 4. Test de Diebold-Mariano (HAC Newey-West, UNILATERAL)

Le test de **Diebold-Mariano** compare, jour a jour, la **perte** de la strategy vis-a-vis de la baseline : `d = baseline - strategy` (positif = la strategy sous-performe). La variance du test est estimee par **Newey-West (HAC) sur les moments bruts** — on ne recentre PAS la serie de difference (recentrer la rendrait a moyenne nulle et le numerateur du test s'annulerait, le rendant inerte).

Le test est **unilateral** : la strategy "gagne" seulement si `d` est significativement **négative** (stat < 0) avec p < 0.05 — un test bilateral seul qualifierait une strategy *significativement pire* comme un succes.


In [8]:
"""Test de Diebold-Mariano, variance HAC Newey-West (Bartlett), sur d = baseline - strategy."""
def dm_test_hac(d: np.ndarray, maxlag: int = 5) -> tuple:
    d = np.asarray(d, dtype=float)
    n = len(d)
    dbar = d.mean()                                   # numerateur : pas recentre
    gamma0 = (d ** 2).mean()                          # moments bruts
    var = gamma0
    for k in range(1, maxlag + 1):
        gamma_k = (d[k:] * d[:-k]).mean()
        var += 2.0 * (1.0 - k / (maxlag + 1)) * gamma_k   # noyau Bartlett
    var /= n
    if var <= 0:
        return 0.0, 1.0
    stat = dbar / np.sqrt(var)
    p = 2.0 * (1.0 - stats.norm.cdf(abs(stat)))       # p bilaterale (lue ci-dessous en unilatereal)
    return float(stat), float(p)

d_loss = (ew_ret - strat_ret).dropna().values         # positif = strategy sous-performe
dm_stat, dm_p = dm_test_hac(d_loss, maxlag=5)
# strategy "gagne" si la perte est significativement NEGATIVE :
dm_wins = (dm_stat < 0.0) and (dm_p < 0.05)

print(f"DM (d = baseline - strategy) : stat = {dm_stat:.3f} | p bilaterale = {dm_p:.4f}")
print(f"Stat < 0 (strategy meilleure) : {dm_stat < 0.0}  |  p < 0.05 : {dm_p < 0.05}")
print(f"DM en faveur de la strategy   : {dm_wins}")


DM (d = baseline - strategy) : stat = 3.647 | p bilaterale = 0.0003
Stat < 0 (strategy meilleure) : False  |  p < 0.05 : True
DM en faveur de la strategy   : False


## 5. Verdict (honnete, net de couts)

Le verdict croise les conditions du critere C : (1) delta OOS moyen **positif**, (2) au moins 3 folds chronologiques positifs, (3) edge cross-seed **>= 2 sigma**, (4) DM **unilateral** en faveur de la strategy (p < 0.05). Sans la conjonction : `NO BEATS` (delta negatif) ou `INCONCLUSIVE`.


In [9]:
"""Verdict honnete — conjonction des 4 conditions du critere C, net de couts."""
cond = {
    "delta OOS moyen > 0": mean_oos_delta > 0,
    f">= 3 folds positifs ({n_positive_folds}/{len(wf_df)})": n_positive_folds >= 3,
    "edge cross-seed >= 2 sigma": (not np.isnan(edge_sigma)) and edge_sigma >= 2.0,
    "DM unilatereal en faveur de la strategy": dm_wins,
}
for k, v in cond.items():
    print(f"  [{'x' if v else ' '}] {k}")

if mean_oos_delta < 0:
    verdict = "NO BEATS"
elif all(cond.values()):
    verdict = "BEATS"
else:
    verdict = "INCONCLUSIVE"

print(f"\nVERDICT : {verdict}  (net de {COST_BPS} bps, univers {N_UNIV} ETF)")


  [ ] delta OOS moyen > 0
  [ ] >= 3 folds positifs (0/5)
  [ ] edge cross-seed >= 2 sigma
  [ ] DM unilatereal en faveur de la strategy

VERDICT : NO BEATS  (net de 10 bps, univers 20 ETF)


### Lecture du verdict

La cellule précédente affiche `VERDICT : NO BEATS`. Les quatre conditions du critère C sont vides :

| Condition | Sortie | Mesure |
|---|---|---|
| delta OOS moyen > 0 | non respectée | -0.1134 % par periode (section 3) |
| au moins 3 folds positifs | non respectée | 0 fold sur 5 |
| edge cross-seed >= 2 sigma | non respectée | -16.43 sigma (section 3) |
| DM unilateral en faveur de la strategy | non respectée | stat = 3.647, du côté de la baseline (section 4) |

**La fenêtre de mesure.** Les prix couvrent du 2 janvier 2015 au 29 juin 2026, soit 2888 séances sur
20 ETF (deux tickers déclarés ne se résolvent pas et sont retirés). Les rendements commencent le
5 janvier 2015 et finissent le 29 juin 2026 : 2887 jours. Le portefeuille est détenu 2880 de ces
jours ; les 7 autres précèdent le premier classement (il faut 5 jours de rendement pour classer) et
suivent l'échéance de la dernière position.

**La convention de détention.** Un classement est posé tous les 5 jours et sa position est reportée
jusqu'au rééquilibrage suivant : le turnover n'est donc non nul qu'aux 576 rééquilibrages, plus une
liquidation de fin de série — 577 jours de turnover non nul. C'est ce report qui rend le portefeuille
investi en permanence. Sans lui, chaque poids ne vivait qu'une seule ligne : le backtest ne
produisait du P&L qu'un jour sur cinq, tout en payant l'entrée puis la sortie des le lendemain, soit
deux allers-retours par cycle au lieu d'un.

**1. La stratégie perd, la baseline gagne.** Net de coûts, la stratégie rapporte -14.35 % par an ;
le portefeuille equal-weight de l'univers, +14.22 % (section 2). L'écart de -0.1134 % par periode se
retrouve sur les cinq folds chronologiques : les cinq `delta_mean` vont de -0.001775 à -0.000297,
tous négatifs, et sur les cinq graines, de -0.1276 % à -0.1109 % : aucune période, aucun
reclassement ne fait exception. Le test de Diebold-Mariano va dans le même sens : la statistique
3.647 est du signe contraire à celui qu'exige la condition (p = 0.0003), la stratégie sous-performe la
baseline de façon significative.

**2. Les deux membres de la comparaison n'ont pas la même exposition.** La stratégie est long-short :
longue du quintile des perdants, courte de celui des gagnants, avec une exposition nette au marché
nulle. La baseline est long-only, investie à 100 % dans l'univers. Sur la période, le portefeuille
equal-weight a monté : une stratégie neutre au marché n'en garde rien, et le delta mélange donc deux
choses, le rendement du marché et la valeur propre du signal. Le verdict répond à la question posée
(la stratégie bat-elle le portefeuille equal-weight ?) ; il ne mesure pas si le reversal a, en soi,
une valeur.

**3. Les coûts de transaction dépassent le P&L brut.** La stratégie est rééquilibrée tous les
`HORIZON` = 5 jours : 576 dates de rééquilibrage (section 1). Chaque cycle achète le portefeuille
puis le revend, soit un turnover de 2 à l'entrée et de 2 à la sortie, 40 bps par cycle au tarif de
`COST_BPS` = 10 bps. Le report du poids entre deux rééquilibrages divise les coûts par deux — 15.01 %
par an — et le P&L brut, lui, ne représente plus qu'environ +0.66 % par an : la différence entre ce
coût et ce gain donne le net de -14.35 %. Le signe du verdict ne dépend pas de cette convention : la
stratégie perd de l'argent dans les deux. Le report ne se contente pas de diviser les coûts par deux ;
il fait surtout tomber le P&L brut, parce que l'essentiel de l'edge du signal tient au premier jour
qui suit le rééquilibrage, et les quatre jours détenus en plus rendent moins qu'ils ne coûtent. Le
`NO BEATS` reste donc un résultat honnête sur cet univers, et non l'artefact d'un backtest qui ne
détenait rien.

**Portée du verdict.** Il vaut pour cet univers (20 ETF chargés sur 22 déclarés), ce classement
(5 jours, quintiles), ce tarif de coûts (10 bps) et cette convention de détention, sur une seule
période. Il est net, et il ne dit pas que le reversal à court terme n'existe pas : il dit que ce
portefeuille, rééquilibré chaque semaine à ce tarif, ne couvre pas ses coûts. L'exercice 3
(long-only, sans jambe courte) et une détention effective sur toute la semaine sont deux façons de
poser autrement la question.


## 6. Exercices d'extension


### Exercice 2 -- Scanner l'horizon

Parcourir `HORIZON` dans `[3, 5, 10, 21]` jours et relever pour chacun le ret annuelise net de couts. Identifier l'horizon ou l'edge est maximal et discuter pourquoi le reversal s'efface aux horizons longs.


In [10]:
# TODO etudiant: pour h in [3, 5, 10, 21], recalculer weights(h) et le ret annuelise net
# result_ex2 = None  # TODO etudiant: dict {horizon: ret_annuelise}
print("Exercice a completer")


Exercice a completer


### Exercice 3 -- Long-only, sans short

Restreindre le portefeuille au **long-only** du quantile bas (les perdants), sans la jambe court. Comparer le ret et la volatility au cas long-short : le reversal survit-il sans levier court ?


In [11]:
# TODO etudiant: zeroer la jambe court des poids, recalculer ret et volatility
print("Exercice a completer")


Exercice a completer


## Conclusion

Ce carnet mesure l'anomalie de reversal a court terme sur un univers d'ETF liquides, out-of-sample (**walk-forward 5-fold chronologique**, multi-seed par bruit de rang), net de 10 bps de couts, avec un test de Diebold-Mariano a variance HAC **unilateral**. Le verdict est **honnete** : il documente ce que l'edge vaut reellement sur cet univers, sans le sur-vendre. Deux points methodologiques sont porteurs : le walk-forward est chronologique (un melange des dates ne laisse aucun test vraiment out-of-sample) et le multi-seed est un bruit de rang reel (un shuffle puis re-tri deterministe ne ferait varier aucun seed).
